# 运行 FastMCP 服务器
了解如何使用各种传输协议（如 STDIO、Streamable HTTP 和 SSE）运行和部署 FastMCP 服务器。

FastMCP 服务器可以根据您的应用程序需求以不同的方式运行，从本地命令行工具到持久化 Web 服务。本指南介绍运行服务器的主要方法，重点介绍可用的传输协议：`STDIO`、`Streamable HTTP` 和 `SSE`。

## run()​方法
`run()`可以通过调用实例上的方法直接从 Python 运行 FastMCP 服务器FastMCP。

> 为了最大程度地提高兼容性，最佳做法是将run()调用放在块中if __name__ == "__main__":。这样可以确保服务器仅在直接执行脚本时启动，而不是在作为模块导入时启动。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="MyServer")

@mcp.tool()
def hello(name: str) -> str:
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

您现在可以通过执行`python my_server.py`来运行此 MCP 服务器。

MCP 服务器可以根据您的应用程序需求，使用多种不同的传输选项运行。该`run()`方法可以接受一个transport参数和其他特定于传输的关键字参数来配置服务器的运行方式。

## FastMCP CLI
FastMCP 还提供了一个命令行界面，无需修改源代码即可运行服务器。安装 FastMCP 后，您可以直接从命令行运行服务器：

In [ ]:
# astmcp run server.py

> 重要：使用 fastmcp run 时，它会完全忽略 if __name__ == "__main__" 块。相反，它会查找名为 mcp、server 或 app 的 FastMCP 对象，并直接调用其 run() 方法和您指定的传输选项。

这意味着您可以使用 fastmcp run 来覆盖代码中指定的传输，这对于测试或更改部署方法而无需修改代码特别有用。

您可以指定传输选项和其他配置：

fastmcp run server.py --transport sse --port 9000

在开发和测试中，您可以使用 `dev` 命令通过 `MCP` 检查器运行服务器：

## transport 选择
以下是可用交通选项的比较，可帮助您选择最适合您需求的transport 选项：

| Transport | 用例 | 推荐 |
| :--- | :--- | :--- |
| STDIO | 本地工具、命令行脚本以及与 Claude Desktop 等客户端的集成 | 最适合本地工具以及客户端管理服务器进程的情况 |
| Streamable HTTP | 基于 Web 的部署、微服务、通过网络公开 MCP | 基于 Web 的部署的推荐选择 |
| SSE | 依赖 SSE 的现有基于 Web 的部署 | 已弃用－新项目优先使用 Streamable HTTP |

## 标准输出
STDIO 传输是本地 MCP 服务器执行的默认且兼容性最广泛的选项。它非常适合本地工具、命令行集成以及 Claude Desktop 等客户端。然而，它的缺点是必须在本地运行 MCP 代码，这可能会给第三方服务器带来安全隐患。

STDIO 是默认传输方式，因此调用`run()`时无需指定。但是，您可以明确指定它，以明确您的意图：

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP()

if __name__ == "__main__":
    mcp.run(transport="stdio")

使用 Stdio 传输时，通常不会将服务器作为单独的进程运行。相反，客户端会为每个会话启动一个新的服务器进程。因此，无需进行任何额外的配置。

## 可流式传输的 HTTP

`Streamable HTTP` 是一种现代、高效的传输协议，用于通过 HTTP 公开您的 MCP 服务器。它是 Web 部署的推荐传输协议。

要使用 `Streamable HTTP` 运行服务器，可以使用` run()` 方法，并将传输参数设置为 `"streamable-http"`。这将在默认主机（127.0.0.1）、端口（8000）和路径（/mcp）上启动 Uvicorn 服务器。

In [ ]:
## server
from fastmcp import FastMCP

mcp = FastMCP()

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

In [ ]:
## client
import asyncio
from fastmcp import Client

async def example():
    async with Client("http://127.0.0.1:8000/mcp") as client:
        await client.ping()

if __name__ == "__main__":
    asyncio.run(example())


要自定义主机、端口、路径或日志级别，请为run()该方法提供适当的关键字参数。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP()

if __name__ == "__main__":
    mcp.run(
        transport="streamable-http",
        host="127.0.0.1",
        port=4200,
        path="/my-custom-path",
        log_level="debug",
    )

In [ ]:
import asyncio
from fastmcp import Client

async def example():
    async with Client("http://127.0.0.1:4200/my-custom-path") as client:
        await client.ping()

if __name__ == "__main__":
    asyncio.run(example())

## SSE

服务器发送事件 (SSE) 是一种基于 HTTP 的协议，用于服务器到客户端的流式传输。虽然 FastMCP 仍然支持 SSE，但它已被弃用，新项目更倾向于使用 Streamable HTTP。

要使用 SSE 运行服务器，可以使用 run() 方法，并将传输参数设置为 "sse"。这将在默认主机（127.0.0.1）、端口（8000）、默认 SSE 路径（/sse）和消息路径（/messages/）上启动 Uvicorn 服务器。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP()

if __name__ == "__main__":
    mcp.run(transport="sse")

In [ ]:
import asyncio
from fastmcp import Client
from fastmcp.client.transports import SSETransport

async def example():
    async with Client(
        transport=SSETransport("http://127.0.0.1:8000/sse")
    ) as client:
        await client.ping()

if __name__ == "__main__":
    asyncio.run(example())

> 请注意，上面示例中的客户端使用显式方式SSETransport连接到服务器。FastMCP 将尝试根据提供的配置推断适当的传输方式，但 HTTP URL 被假定为` Streamable HTTP`（从 FastMCP 2.3.0 开始）。

要自定义主机、端口或日志级别，请为`run()`该方法提供适当的关键字参数。您还可以调整 SSE 路径（客户端应连接到的路径）和消息 POST 端点（客户端用于发送后续消息的端点）。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP()

if __name__ == "__main__":
    mcp.run(
        transport="sse",
        host="127.0.0.1",
        port=4200,
        log_level="debug",
        path="/my-custom-sse-path",
    )

In [ ]:
import asyncio
from fastmcp import Client
from fastmcp.client.transports import SSETransport

async def example():
    async with Client(
        transport=SSETransport("http://127.0.0.1:4200/my-custom-sse-path")
    ) as client:
        await client.ping()

if __name__ == "__main__":
    asyncio.run(example())

## 异步使用
FastMCP 提供了同步和异步 API 来运行服务器。前面示例中的`run()`方法是同步方法，其内部`anyio.run()`用于运行异步服务器。对于已经在异步上下文中运行的应用程序，FastMCP 也提供了`run_async()`异步方法。

In [ ]:
from fastmcp import FastMCP
import asyncio

mcp = FastMCP(name="MyServer")

@mcp.tool()
def hello(name: str) -> str:
    return f"Hello, {name}!"

async def main():
    # Use run_async() in async contexts
    await mcp.run_async(transport="streamable-http")

if __name__ == "__main__":
    asyncio.run(main())

该`run()`方法无法在异步函数内部调用，因为它已经在内部创建了自己的异步事件循环。如果您尝试`run()`在异步函数内部调用，则会收到一条错误消息，提示事件循环已在运行。

始终`run_async()`在异步函数内部和`run()`同步上下文中使用。

`run()`和都`run_async()`接受相同的传输参数，因此上述所有示例都适用于这两种方法。

## 自定义路线
您还可以向 FastMCP 服务器添加自定义 Web 路由，这些路由将与 MCP 端点一起公开。为此，请使用`@custom_route`装饰器。请注意，这比使用完整的 ASGI 框架灵活性较低，但对于向独立服务器添加简单的端点（例如健康检查）非常有用。




In [ ]:
from fastmcp import FastMCP
from starlette.requests import Request
from starlette.responses import PlainTextResponse

mcp = FastMCP("MyServer")

@mcp.custom_route("/health", methods=["GET"])
async def health_check(request: Request) -> PlainTextResponse:
    return PlainTextResponse("OK")

if __name__ == "__main__":
    mcp.run()